# Device mapping

Put device names in element alias field.

The master mapping of device name to element name is from the SLACPROD Oracle Database, downloaded as a CSV file:

https://oraweb.slac.stanford.edu/apex/slacprod/f?p=116:600


In [1]:
import pandas as pd
import numpy as np
import json
import os

# SLACPROD Oracle Table

In [2]:
#%env LCLS_LATTICE=/home/mpe/repos/lcls-lattice
os.path.expandvars('$LCLS_LATTICE')

'/home/mpe/repos/lcls-lattice/'

In [3]:
# Table extracted from SLACPROD Oracle Database
#%env LCLS_LATTICE=/home/mpe/code/lcls-lattice
MASTER = '$LCLS_LATTICE/bmad/conversion/from_oracle/lcls_elements.csv'

df = pd.read_csv(os.path.expandvars(MASTER))
# Remove empty
df = df[['Element', 'Control System Name']].dropna()

KeyError: "None of [Index(['Element', 'Control System Name'], dtype='object')] are in the [columns]"

In [4]:
# These are unique
MADNAMES = list(df['Element'])
len(MADNAMES), len(set(MADNAMES))

(3700, 3700)

In [5]:
# These are not
DEVICENAMES = list(df['Control System Name'])
len(DEVICENAMES), len(set(DEVICENAMES))

(3700, 3287)

In [6]:
# These devices have multiple elements - a mistake?
series  = df.groupby('Control System Name')['Element'].apply(list)
for i, val in series.items():
    if len(val) > 1:
        # Skip klystrons - these are expected to be duplicated
        if not val[0].startswith('K'):
            print(i, val)

In [7]:
# dict for lookup
DEVICE = dict(zip(MADNAMES, DEVICENAMES))
json.dump(DEVICE, open('element_devices.json', 'w'))

# Models 

In [8]:
BDIR = os.path.expandvars('$LCLS_LATTICE/bmad/')

In [9]:
# All models
MODELS = [d for d in os.listdir(BDIR+'models/') if os.path.isdir(BDIR+'/models/'+d)]
INITFILE = {model:f'$LCLS_LATTICE/bmad/models/{model}/tao.init' for model in MODELS}
INITFILE

{'sc_bsyd': '$LCLS_LATTICE/bmad/models/sc_bsyd/tao.init',
 'hxr': '$LCLS_LATTICE/bmad/models/hxr/tao.init',
 'sc_sxr': '$LCLS_LATTICE/bmad/models/sc_sxr/tao.init',
 'sc_inj': '$LCLS_LATTICE/bmad/models/sc_inj/tao.init',
 'lcls_complex': '$LCLS_LATTICE/bmad/models/lcls_complex/tao.init',
 'cu_hxr': '$LCLS_LATTICE/bmad/models/cu_hxr/tao.init',
 'sc_diag0': '$LCLS_LATTICE/bmad/models/sc_diag0/tao.init',
 'cu_linac': '$LCLS_LATTICE/bmad/models/cu_linac/tao.init',
 'sc_hxr': '$LCLS_LATTICE/bmad/models/sc_hxr/tao.init',
 'sc_dasel': '$LCLS_LATTICE/bmad/models/sc_dasel/tao.init',
 'cu_sxr': '$LCLS_LATTICE/bmad/models/cu_sxr/tao.init',
 'cu_spec': '$LCLS_LATTICE/bmad/models/cu_spec/tao.init',
 'cu_inj': '$LCLS_LATTICE/bmad/models/cu_inj/tao.init'}

In [10]:
# Tack on FACET-II if availiable

FDIR = os.path.expandvars('$FACET2_LATTICE/bmad/')

if os.path.exists(FDIR):
    print('Adding FACET-II')
    model = 'f2_elec'
    ifile = os.path.join(FDIR, f'models/{model}/tao.init')
    if os.path.exists(ifile):
        print(f'Adding {model} model')
        MODELS.append(model)
        INITFILE[model] = ifile

Adding FACET-II
Adding f2_elec model


# PyTao

In [11]:
from pytao import Tao
import os

In [12]:
#init = INITFILE['sc_dasel']
#print(init)
#tao = Tao(f'-init {init} -noplot')

In [13]:
def ele_names(model):
    init = INITFILE[model]
    print(f'{model}')
    tao = Tao(f'-init {init} -noplot')
    names = tao.cmd('python lat_list 1@0>>*|model ele.name')
    return names

def remove_superslaves(names):
    return [x for x in names if '#' not in x]

In [14]:
def write_devicenames(unames, filename):
    my_names = remove_superslaves(unames)
    lines = ['! ---------',
             '! Device mapping derived from '+MASTER
             
            ]
    for name in my_names:
        if name in DEVICE:
            line = name+'[alias]='+ DEVICE[name]
            
        else:
            #continue
            line = '! No device listed for: '+name
        lines.append(line)    
    with open(filename, 'w') as f:
        for line in lines:
            f.write(line+'\n')
    print('Written:', filename)

# Add to CU Master

In [15]:
# Output filename
CU_FILE = f'{BDIR}/master/LCLScu_devicenames.bmad'
CU_FILE_BAK = f'{BDIR}/master/LCLScu_devicenames-bak.bmad'
os.rename(CU_FILE,CU_FILE_BAK)
open(CU_FILE, 'a').close()  #make an empty file

In [16]:
models = ['cu_hxr', 'cu_sxr', 'cu_spec']
names = []
for m in models:
    print(m)
    names += ele_names(m)
unames = sorted(list(set(names)))

cu_hxr
cu_hxr
cu_sxr
cu_sxr
cu_spec
cu_spec


In [17]:
write_devicenames(unames, CU_FILE)

Written: /home/mpe/temp_repo/lcls-lattice//bmad//master/LCLScu_devicenames.bmad


# Add to SC Master

In [18]:
SC_FILE = f'{BDIR}/master/LCLSsc_devicenames.bmad'

In [19]:
models = ['sc_hxr', 'sc_sxr', 'sc_diag0', 'sc_bsyd', 'sc_dasel']
names = []
for m in models:
    print(m)
    names += ele_names(m)
unames = sorted(list(set(names)))

sc_hxr
sc_hxr
sc_sxr
sc_sxr
sc_diag0
sc_diag0
sc_bsyd
sc_bsyd
sc_dasel
sc_dasel


In [20]:
write_devicenames(unames, SC_FILE)

Written: /home/mpe/temp_repo/lcls-lattice//bmad//master/LCLSsc_devicenames.bmad


# Add to FACET-II

In [21]:
if os.path.exists(FDIR):
    F2_FILE = f'{FDIR}/master/FACET2e_devicenames.bmad'
    models = ['f2_elec']
    names = []
    for m in models:
        print(m)
        names += ele_names(m)
    unames = sorted(list(set(names)))
    
    write_devicenames(unames, F2_FILE)

f2_elec
f2_elec
Written: /home/mpe/repos/facet2-lattice//bmad//master/FACET2e_devicenames.bmad


# elementdevices (old method)

In [22]:
#
#
#    ELEMENTDEVICES = os.path.expandvars('$LCLS_LATTICE/mad/elementdevices.dat')
#    os.path.exists(ELEMENTDEVICES)
#    
#    def parse_elementdevices(elementdevices_filename):
#        """
#        
#        Parameters
#        ----------
#        elementdevices_filename
#        
#        Returns
#        -------
#        device: dict of ele_name:devicename
#        not_found: list of ele_names with no device
#        
#        """
#        device = {}
#        not_found = []
#        with open(elementdevices_filename) as f:
#            for line in f:
#                x = line.split()
#                if len(x) != 2:
#                    continue
#    
#                ele, devicename = x
#                if devicename == '-':
#                    not_found.append(ele)
#                    continue
#                if ele in device:
#                    raise ValueError('ele already has a a device:', ele, device[ele])
#    
#                device[ele] = devicename   
#                
#        return device, not_found
#    DNAME, NOT_FOUND = parse_elementdevices(ELEMENTDEVICES)    
#    len(list(DNAME)), len(NOT_FOUND)

In [23]:
## Check for missing or mismatched items
#for ele, dev in DNAME.items():
#    if ele not in DEVICE:
#        #continue
#        print('Missing from Oracle table:', ele, dev)
#    else:
#        oracle_dev = DEVICE[ele] 
#        if oracle_dev != dev:
#        #    continue
#            print('Device mismatch for ele, oracle, elementdevices.dat:', ele, oracle_dev, dev)